# External Monitoring

Say you have a complicated agentic workflow or are running in a restricted environment—whatever the reason, you may not be able to just lift an application or workflow and bring it onto DataRobot to run. For these scenarios, you can still have "bolt-on" monitoring and observability by using DataRobot's agent libraries.

The decorator in `monitoring.py` allows you to monitor any agentic or generative workflow directly using DataRobot's external monitoring.

**11.1 Updates**
Note that in 11.1, we will also be supporting [OpenTelemetry](https://opentelemetry.io/) for monitoring, helping make this process even easier.

## Set-up

The first step is registering an "External Model" with DataRobot and creating a deployment for it. This is giving DataRobot the metadata it needs to start collecting your telemetry data. We will demonstrate this by building a chatbot that helps us discuss sonnets from William Shakespeare.

In [ ]:

import datarobot as dr
from datarobot.models import RegisteredModelVersion
import pandas as pd
dr_client = dr.Client()
NAME = "Shakespeare External Model"
INPUT_PROMPT_NAME = "promptText"
OPENAI_KEY = '''<API_KEY>'''  # Replace with your OpenAI API key

In [28]:
#Obtain the Shakespear Base Data

import requests as req

sonnets = req.get("https://raw.githubusercontent.com/enerrio/Generate-Shakespeare-Sonnets/refs/heads/master/sonnets.txt").text.split("\n\n")

dataset = dr.Dataset.create_from_in_memory_data(pd.DataFrame([(s,s) for s in sonnets], columns=[INPUT_PROMPT_NAME, "output"]), fname="Shakespear Sonnets")

In [3]:
# create the external model
external_model = dr_client.post(
    "modelPackages/fromJSON",
    data={
        "name": NAME,
        "modelDescription": {"description": "An external model that generates text analyze sonnets of Shakespeare."},
        "target": {"type": "TextGeneration", "name": "output"},
        "textGeneration": {
            "prompt": INPUT_PROMPT_NAME,
        }, 
        "datasets": {
            "training_data_catalog_id": dataset.id
        }
    }
).json()



Now we also have to create metadata for where this model runs and deploy it. 

In [4]:
pred_environment = dr.PredictionEnvironment.create(
    name="Shakespeare External Model Environment",
    platform=dr.PredictionEnvironmentPlatform.OTHER
)

deployment = dr.Deployment.create_from_registered_model_version(external_model.get("id"),
    label="Shakespeare External Model Deployment",
prediction_environment_id=pred_environment.id)

deployment.update_association_id_settings(column_names=["id"], required_in_prediction_requests=False)
deployment.update_predictions_data_collection_settings(enabled=True)
deployment.update_drift_tracking_settings(target_drift_enabled=False, feature_drift_enabled=True)


In [5]:
deployment.id

'68430b7432a87516e04432c5'

Now let's create our analysis app. 

In [29]:
from openai import OpenAI
from random import choice

oai_client = OpenAI(api_key=OPENAI_KEY)

import monitoring
from importlib import reload
reload(monitoring)


@monitoring.dr_monitor(deployment_id=deployment.id, model_id=external_model.get("modelId"), datarobot_api_token=dr_client.token, datarobot_endpoint=dr_client.endpoint)
def describe_sonnet(sonnet = choice(sonnets), id=None):
    response = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """You are a helpful assistant. 
             Pretend you are an expert in Shakespeare's works."""},
            {"role": "user", "content": f"Describe the following sonnet:\n\n{sonnet}"}
        ],
    )
    resp = response.choices[0].message.content.strip()
    print(resp)
    return resp

r = describe_sonnet()


This sonnet is a beautiful expression of love and fidelity that deftly navigates themes of absence, longing, and the unwavering nature of true affection. It appears to be an offering to a beloved, addressing concerns of fidelity and the anxiety that physical separation can instigate in a relationship.

The speaker begins with a strong declaration, "O! never say that I was false of heart," insisting that despite their physical distance, their feelings remain true and steadfast. The phrase "absence seemed my flame to qualify" suggests that while separation may cool the intensity of their love, it does not diminish its existence.

The speaker uses metaphors involving travel and home to depict their emotional reality. The idea of departing from oneself mirrors the fear of losing this connection, emphasizing that their love resides in the beloved's heart—"my soul which in thy breast doth lie." Thus, the beloved is portrayed as the true home of the speaker's love.

The tone shifts as the spe

In [19]:
from IPython.display import display, Markdown
number_of_requests = deployment.get_service_stats().metrics['totalPredictions']

display(Markdown(f'''The workflow has been called {number_of_requests} times.'''))

The workflow has been called 4 times.

## Custom Metrics

It's also possible to log custom metrics. Here we can use the [answer relevancy](https://docs.llamaindex.ai/en/stable/examples/evaluation/answer_and_context_relevancy/) from LLama Index as a custom metric which we can log as well. 



In [43]:
custom_metric = dr.models.deployment.CustomMetric.create(
    name="LLama Answer Relevance",
    description="A custom metric to evaluate the relevance of the answers provided by the model.",
    units="score",
    is_model_specific=False, 
    aggregation_type=dr.enums.CustomMetricAggregationType.AVERAGE,
    directionality=dr.enums.CustomMetricDirectionality.HIGHER_IS_BETTER,
    deployment_id=deployment.id,
    
)

In [70]:
from uuid import uuid4
from llama_index.core.evaluation import AnswerRelevancyEvaluator
from llama_index.llms.openai import OpenAI as LlamaOpenAI


CONVERSATION_ID = f"""Relevance-{str(uuid4())[0:5]}"""
sonnet = choice(sonnets)

response = describe_sonnet(sonnet=sonnet, id=CONVERSATION_ID)

evaluator = AnswerRelevancyEvaluator(llm=LlamaOpenAI(model="gpt-4o-mini", temperature=0.0, api_key=OPENAI_KEY))

score = evaluator.evaluate(query=sonnet, response=response)





This sonnet is one of Shakespeare's famous "Fair Youth" sonnets, specifically Sonnet 17, which is part of a sequence often interpreted as addressing a young man of great beauty and promise. The poem deals with themes of immortality through poetry, the nature of beauty, and the challenge of adequately capturing that beauty in verse.

The speaker begins with a rhetorical question, expressing doubt about whether his poetry will be believed in the future if it cannot fully capture the true worth of the subject’s beauty (“your most high deserts”). The metaphor of a tomb suggests that his verse, while an attempt to preserve and convey the subject's life and essence, is ultimately inadequate (“hides your life”). 

The speaker laments that even if he could perfectly describe the beauty of the young man's eyes and all his graces, future generations might dismiss his work as exaggerated or false (“Such heavenly touches ne'er touched earthly faces”), revealing the tension between the idealized be

In [71]:
custom_metric.submit_values(data=[
    {
            "metric_name": custom_metric.name,
    "metric_id": custom_metric.id,
    "deployment_id": deployment.id,
    "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    "association_id": CONVERSATION_ID,
    "value": score.score,
    "sample_size": 1,
    }
])

In [77]:
from datetime import datetime, timedelta


end = datetime.now()

custom_metric.get_summary(end=end, start=end - timedelta(days=1)).metric['value']

### Cleanup

In [ ]:
deployment.delete()
pred_environment.delete()
dr_client.delete(f"registeredModels/{external_model.get("registeredModelId")}")



In [17]:
r

'This sonnet is one of Shakespeare\'s 153 sonnets, specifically Sonnet 40. In this poem, the speaker expresses profound emotional turmoil due to an unreciprocated love that not only affects him but also deeply impacts his friend. \n\nThe sonnet opens with a curse upon the heart that causes the speaker pain, as it leads to anguish not just for himself, but also for his beloved friend. The speaker wonders why it is not sufficient for him to suffer alone; instead, he feels compelled to share the burden of this "slavery" with a friend who is affected by the same allure of the beloved.\n\nThe second quatrain reveals the deep sense of loss and betrayal the speaker feels. The "cruel eye" of the beloved has taken away his sense of self, leaving him feeling forsaken. There’s a feeling of being tormented thrice over: once for himself, once for his friend, and once again due to the shared bond of suffering.\n\nIn the third quatrain, the speaker suggests that even if he must remain confined to the